# Phase 6a — Le SFT : de « continuer un article » à « répondre »

Ton modèle sait écrire du français, mais il ne sait que **continuer** un texte. Si tu lui écris « Quelle est la capitale de l'Italie ? », il va continuer comme si c'était le début d'un article.

Le **SFT** (*Supervised Fine-Tuning*) corrige ça : on lui montre des milliers d'exemples « instruction → réponse », et il apprend le **format d'une conversation**. On ne lui apprend presque rien de nouveau sur le monde : on lui apprend un **rôle**.

**Sur Kaggle**, dans le panneau de droite (Settings) :
- **Accelerator : GPU T4 x2** (on n'en utilisera qu'un) ;
- **Internet : On** (pour télécharger ton modèle depuis Hugging Face).

L'entraînement est court : une vingtaine de minutes environ.

In [1]:
!pip -q install tokenizers safetensors huggingface_hub
import os, sys, json, math, time, random
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from huggingface_hub import hf_hub_download
from safetensors.torch import load_model, save_model
from tokenizers import Tokenizer

TRAVAIL = "/kaggle/working"
DEPOT = "Ezeck01/petit-gpt-fr-46m"
for f in ["model.py", "config.json", "tokenizer.json", "model.safetensors"]:
    hf_hub_download(DEPOT, f, local_dir=TRAVAIL)
sys.path.append(TRAVAIL)
from model import Config, GPT

device = "cuda"
print("GPU :", torch.cuda.get_device_name(0))
config_base = Config(**json.load(open(f"{TRAVAIL}/config.json")))
base = GPT(config_base)
load_model(base, f"{TRAVAIL}/model.safetensors")
tokenizer = Tokenizer.from_file(f"{TRAVAIL}/tokenizer.json")
print("Ton modèle de base est chargé depuis Hugging Face.")

model.py:   0%|          | 0.00/4.04k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.15M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  186MB            

model.safetensors: downloading bytes:           |  0.00B            

GPU : Tesla T4
Ton modèle de base est chargé depuis Hugging Face.


## 1. Les données

`French-Alpaca-dataset-Instruct-110K` (Jonathan Pacifico, licence Apache 2.0) : 110 000 instructions en français, avec réponse. Chaque exemple a trois champs :
- `instruction` : la consigne ;
- `input` : un contexte éventuel (souvent vide) ;
- `output` : la réponse attendue.

⚠️ Les réponses ont été **générées par GPT-3.5**, pas écrites par des humains. C'est très courant pour le SFT (ça coûte bien moins cher), mais ton modèle va imiter le style de GPT-3.5, et ses erreurs.

In [2]:
chemin = hf_hub_download("jpacifico/French-Alpaca-dataset-Instruct-110K",
                         "Acquiesce_data_110k_instructions.json", repo_type="dataset", local_dir=TRAVAIL)
donnees = json.load(open(chemin, encoding="utf-8"))
print(f"{len(donnees):,} exemples\n")
random.seed(0)
for ex in random.sample(donnees, 4):
    print("INSTRUCTION :", ex["instruction"])
    if ex["input"]:
        print("CONTEXTE    :", ex["input"][:150])
    print("RÉPONSE     :", ex["output"][:200])
    print("-" * 70)

Acquiesce_data_110k_instructions.json: reconstructing file:   0%|          |  0.00B / 52.0MB            

Acquiesce_data_110k_instructions.json: downloading bytes:           |  0.00B            

110,368 exemples

INSTRUCTION : Donne-moi une recette de plat végétarien sain et savoureux.
RÉPONSE     : Salade de quinoa à la mangue et à l'avocat : Ingrédients : - 1 tasse de quinoa - 1 mangue mûre, pelée et coupée en dés - 1 avocat mûr, pelé et coupé en dés - 1/4 tasse d'oignon rouge haché finement - 
----------------------------------------------------------------------
INSTRUCTION : Planifie une journée de randonnée à travers les Alpes suisses, en incluant les points de départ, les points d'arrivée et les étapes intermédiaires.
RÉPONSE     : La randonnée débute à Zermatt, passant par le lac de Moiry, et se termine à Grimentz, avec plusieurs arrêts panoramiques le long du chemin.
----------------------------------------------------------------------
INSTRUCTION : Explique le processus de l'équation quadratique.
RÉPONSE     : L'équation quadratique est une équation de la forme ax^2 + bx + c = 0, avec a, b et c comme constantes connues et x comme variable inconnue. Pour résoudre une

## 2. Le format de conversation et les tokens spéciaux

Il faut que le modèle sache **qui parle**. On crée deux nouveaux tokens :
- `<|utilisateur|>` : ce qui suit est la demande ;
- `<|assistant|>` : ce qui suit est la réponse ;
- et on réutilise `<|endoftext|>` pour dire « la réponse est finie ». Sans lui, le modèle ne saurait jamais s'arrêter.

```
<|utilisateur|>
Quelle est la capitale de l'Italie ?
<|assistant|>
La capitale de l'Italie est Rome.<|endoftext|>
```

**Le problème :** ton modèle a 16 384 tokens, ces deux-là n'existent pas. Il faut **agrandir sa table d'embeddings** de 2 lignes. On initialise ces nouvelles lignes avec la **moyenne** de tous les vecteurs existants : un point de départ neutre, que le SFT va ajuster.

In [3]:
UTIL, ASSIST, FIN = "<|utilisateur|>", "<|assistant|>", "<|endoftext|>"
tokenizer.add_special_tokens([UTIL, ASSIST])
ID_UTIL, ID_ASSIST, ID_FIN = (tokenizer.token_to_id(t) for t in (UTIL, ASSIST, FIN))
print("Nouveaux ids :", ID_UTIL, ID_ASSIST, "| fin :", ID_FIN)

config = Config(**{**json.load(open(f"{TRAVAIL}/config.json")), "vocab_size": tokenizer.get_vocab_size()})
modele = GPT(config)
etat = base.state_dict()
ancienne = etat.pop("emb_tokens.weight")
etat.pop("tete.weight")
nouvelle = torch.cat([ancienne, ancienne.mean(0, keepdim=True).repeat(2, 1)])
modele.load_state_dict({**etat, "emb_tokens.weight": nouvelle, "tete.weight": nouvelle})
modele.to(device)
base.to(device).eval()
print(f"Table d'embeddings : {tuple(ancienne.shape)} -> {tuple(modele.emb_tokens.weight.shape)}")

Nouveaux ids : 16384 16385 | fin : 0
Table d'embeddings : (16384, 512) -> (16386, 512)


## 3. Tokenizer avec masquage

Point clé du SFT : **on n'entraîne le modèle que sur les réponses**.

La question est dans l'entrée (il doit la lire), mais on ne veut pas qu'il apprenne à *écrire* des questions. On met donc la cible à **−100** sur toute la partie utilisateur : PyTorch ignore ces positions dans le calcul de la perte (c'est la valeur par défaut d'`ignore_index` dans `cross_entropy`, que ton `model.py` utilise déjà).

On écarte aussi :
- les exemples trop longs (plus de 512 tokens, la taille du contexte) ;
- ceux qui contiennent du code : ton modèle n'en a jamais vu, il ne ferait qu'imiter la forme.

In [4]:
def formater(ex):
    demande = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    return demande.strip(), ex["output"].strip()

def encoder(demande, reponse=None):
    ids = [ID_UTIL] + tokenizer.encode("\n" + demande + "\n").ids + [ID_ASSIST] + tokenizer.encode("\n").ids
    if reponse is None:
        return ids
    ids_reponse = tokenizer.encode(reponse).ids + [ID_FIN]
    cibles = [-100] * len(ids) + ids_reponse        # -100 : ignoré par la perte
    return ids + ids_reponse, cibles

exemples, ecartes = [], 0
for ex in donnees:
    demande, reponse = formater(ex)
    if not reponse or "```" in reponse or "def " in reponse:
        ecartes += 1
        continue
    ids, cibles = encoder(demande, reponse)
    if len(ids) > config.block_size:
        ecartes += 1
        continue
    exemples.append((ids, cibles))

random.shuffle(exemples)
val_ex, train_ex = exemples[:1000], exemples[1000:]
n_tokens = sum(len(i) for i, _ in train_ex)
n_appris = sum(sum(c != -100 for c in cib) for _, cib in train_ex)
print(f"Gardés : {len(exemples):,} | écartés : {ecartes:,}")
print(f"Train : {len(train_ex):,} exemples, {n_tokens:,} tokens dont {n_appris:,} appris ({n_appris / n_tokens:.0%})")

# Visualisation du masque sur un exemple
ids, cibles = encoder("Quelle est la capitale de l'Italie ?", "La capitale de l'Italie est Rome.")
print("\nToken             | appris ?")
for i, c in zip(ids, cibles):
    print(f"{tokenizer.decode([i], skip_special_tokens=False)!r:<18}| {'✅' if c != -100 else '—'}")

Gardés : 109,884 | écartés : 484
Train : 108,884 exemples, 11,238,399 tokens dont 8,024,217 appris (71%)

Token             | appris ?
'<|utilisateur|>' | —
'\n'              | —
'Q'               | —
'uelle'           | —
' est'            | —
' la'             | —
' capitale'       | —
' de'             | —
' l'              | —
"'"               | —
'Italie'          | —
' ?'              | —
'\n'              | —
'<|assistant|>'   | —
'\n'              | —
'La'              | ✅
' capitale'       | ✅
' de'             | ✅
' l'              | ✅
"'"               | ✅
'Italie'          | ✅
' est'            | ✅
' Rome'           | ✅
'.'               | ✅
'<|endoftext|>'   | ✅


Remarque : la cible est **décalée d'un cran** par rapport à l'entrée (comme au pré-entraînement). Dans l'affichage, « appris » veut dire « ce token fait partie de ce que le modèle doit produire ». Le dernier token appris est `<|endoftext|>` : le modèle apprend aussi **à s'arrêter**.

## 4. Les lots et la perte de départ

Les exemples n'ont pas tous la même longueur. Dans un lot, on complète les plus courts (*padding*) jusqu'à la longueur du plus long, avec des cibles à −100 pour que le remplissage ne compte pas.

On mesure la perte sur les réponses de validation **avant** le SFT : c'est la référence.

In [5]:
TAILLE_LOT = 32

def faire_lot(exs):
    L = max(len(i) for i, _ in exs)
    x = torch.full((len(exs), L), ID_FIN, dtype=torch.long)
    y = torch.full((len(exs), L), -100, dtype=torch.long)
    for k, (ids, cib) in enumerate(exs):
        x[k, :len(ids)] = torch.tensor(ids)
        y[k, :len(cib) - 1] = torch.tensor(cib[1:])     # décalage d'un cran
    return x.to(device), y.to(device)

@torch.no_grad()
def perte_val(m):
    m.eval()
    total, n = 0.0, 0
    for k in range(0, len(val_ex), TAILLE_LOT):
        x, y = faire_lot(val_ex[k:k + TAILLE_LOT])
        with torch.autocast("cuda", dtype=torch.float16):
            _, perte = m(x, y)
        nb = (y != -100).sum().item()
        total += perte.item() * nb
        n += nb
    m.train()
    return total / n

perte_avant = perte_val(modele)
print(f"Perte sur les réponses, AVANT le SFT : {perte_avant:.3f}")

Perte sur les réponses, AVANT le SFT : 2.960


## 5. L'entraînement

Mêmes ingrédients qu'au pré-entraînement, avec deux différences :
- un **taux d'apprentissage plus bas** (2e-4 au lieu de 6e-4) : on ajuste un modèle qui sait déjà le français, on ne veut pas tout casser ;
- **2 passages** (époques) sur les données : il y a bien moins de texte qu'au pré-entraînement.

On trie les exemples par longueur à l'intérieur de gros paquets, pour que les lots contiennent des longueurs proches : beaucoup moins de remplissage, donc plus rapide.

In [6]:
EPOQUES, LR_MAX, WARMUP = 2, 2e-4, 100

def lots_epoque():
    exs = train_ex[:]
    random.shuffle(exs)
    paquets = [sorted(exs[k:k + 3200], key=lambda e: len(e[0])) for k in range(0, len(exs), 3200)]
    lots = [p[k:k + TAILLE_LOT] for p in paquets for k in range(0, len(p), TAILLE_LOT)]
    random.shuffle(lots)
    return lots

etapes_par_epoque = math.ceil(len(train_ex) / TAILLE_LOT)
ETAPES = EPOQUES * etapes_par_epoque

def taux(e):
    if e < WARMUP:
        return LR_MAX * (e + 1) / WARMUP
    return LR_MAX * 0.5 * (1 + math.cos(math.pi * (e - WARMUP) / (ETAPES - WARMUP)))

optimiseur = torch.optim.AdamW(modele.parameters(), lr=LR_MAX, betas=(0.9, 0.95), weight_decay=0.1, fused=True)
scaler = torch.amp.GradScaler()
modele.train()
etape, t0 = 0, time.time()
for epoque in range(EPOQUES):
    for exs in lots_epoque():
        for g in optimiseur.param_groups:
            g["lr"] = taux(etape)
        x, y = faire_lot(exs)
        with torch.autocast("cuda", dtype=torch.float16):
            _, perte = modele(x, y)
        scaler.scale(perte).backward()
        scaler.unscale_(optimiseur)
        torch.nn.utils.clip_grad_norm_(modele.parameters(), 1.0)
        scaler.step(optimiseur)
        scaler.update()
        optimiseur.zero_grad(set_to_none=True)
        if etape % 200 == 0:
            reste = (ETAPES - etape) * (time.time() - t0) / max(1, etape)
            print(f"époque {epoque + 1} | étape {etape:>5}/{ETAPES} | perte {perte.item():.3f} | reste ~{reste / 60:.0f} min")
        etape += 1
    print(f"\n=== fin de l'époque {epoque + 1} | perte val (réponses) : {perte_val(modele):.3f} ===\n")

perte_apres = perte_val(modele)
print(f"Perte sur les réponses : {perte_avant:.3f} (avant) -> {perte_apres:.3f} (après)")

époque 1 | étape     0/6806 | perte 2.817 | reste ~70 min
époque 1 | étape   200/6806 | perte 2.328 | reste ~9 min
époque 1 | étape   400/6806 | perte 2.138 | reste ~9 min
époque 1 | étape   600/6806 | perte 2.099 | reste ~9 min
époque 1 | étape   800/6806 | perte 2.150 | reste ~9 min
époque 1 | étape  1000/6806 | perte 1.570 | reste ~8 min
époque 1 | étape  1200/6806 | perte 1.617 | reste ~8 min
époque 1 | étape  1400/6806 | perte 1.736 | reste ~8 min
époque 1 | étape  1600/6806 | perte 1.819 | reste ~7 min
époque 1 | étape  1800/6806 | perte 1.567 | reste ~7 min
époque 1 | étape  2000/6806 | perte 1.717 | reste ~7 min
époque 1 | étape  2200/6806 | perte 1.713 | reste ~7 min
époque 1 | étape  2400/6806 | perte 1.865 | reste ~6 min
époque 1 | étape  2600/6806 | perte 1.751 | reste ~6 min
époque 1 | étape  2800/6806 | perte 1.568 | reste ~6 min
époque 1 | étape  3000/6806 | perte 1.537 | reste ~5 min
époque 1 | étape  3200/6806 | perte 1.583 | reste ~5 min
époque 1 | étape  3400/6806 | 

## 6. Avant / après

On pose les mêmes questions aux deux modèles :
- **le modèle de base** reçoit la question telle quelle : il fait ce qu'il sait faire, **continuer le texte** ;
- **le modèle SFT** reçoit la question dans le format de conversation, et doit **répondre**, puis s'arrêter.

Attends-toi à ce que les réponses aient **la bonne forme** mais soient souvent **fausses** : 46 M de paramètres, c'est très peu de connaissances. Le SFT apprend le rôle, pas le savoir.

In [7]:
@torch.no_grad()
def generer(m, ids, n=80, temperature=0.7, top_k=40, stop=None):
    m.eval()
    x = torch.tensor([ids], device=device)
    for _ in range(n):
        with torch.autocast("cuda", dtype=torch.float16):
            logits, _ = m(x[:, -config.block_size:])
        logits = logits[0, -1].float() / temperature
        seuil = torch.topk(logits, top_k).values[-1]
        logits[logits < seuil] = -float("inf")
        suivant = torch.multinomial(F.softmax(logits, -1), 1)
        if stop is not None and suivant.item() == stop:
            break
        x = torch.cat([x, suivant[None]], dim=1)
    return tokenizer.decode(x[0, len(ids):].tolist())

questions = [
    "Quelle est la capitale de l'Italie ?",
    "Donne trois conseils pour bien dormir.",
    "Explique ce qu'est un volcan en une phrase.",
    "Écris une phrase avec les mots « soleil » et « mer ».",
    "Quel est le contraire du mot « rapide » ?",
    "Résume l'histoire de la Révolution française en deux phrases.",
]
torch.manual_seed(0)
for q in questions:
    print("❓", q)
    print("   BASE :", generer(base, tokenizer.encode(q).ids, n=50).replace("\n", " ")[:300])
    print("   SFT  :", generer(modele, encoder(q), stop=ID_FIN).strip().replace("\n", " / "))
    print()

❓ Quelle est la capitale de l'Italie ?
   BASE : )  Les villes du Grand Orient du Sud (1890-1972)  Durant les années 1870, la ville de Rome était divisée en deux entités, la ville de Rome et la ville de Naples, regroupant les villes des deux
   SFT  : La capitale de l'Italie est Rome.

❓ Donne trois conseils pour bien dormir.
   BASE :   Le dernier conseil de la ville est composé de quatre conseillers. Il est composé de trois conseillers. Un représentant est nommé pour la ville de Saint-Jean-de-la-Roche, et un représentant est nommé pour la ville de Saint
   SFT  : 1. Limitez l'exposition aux appareils électroniques lorsqu'ils ne sont pas utilisés. 2. Utilisez des techniques de relaxation comme la respiration profonde ou la relaxation musculaire pour réduire le stress. 3. Limitez la consommation de caféine et d'alcool en général.

❓ Explique ce qu'est un volcan en une phrase.
   BASE :   Cette phrase est aussi appelée « s'appartenant au village »  En français :  L'expression  L'express

## 7. À toi de jouer

Pose tes propres questions. Teste aussi ses limites : une question piège, une question sur un sujet très précis, une demande de calcul…

In [8]:
def demander(question):
    return generer(modele, encoder(question), n=120, stop=ID_FIN).strip()

print(demander("Qui a écrit Les Misérables ?"))

Les Misérables a été écrit par Victor Hugo.


## 8. Sauvegarder et publier

On publie dans un **nouveau dépôt** (`...-instruct`), en privé. Le modèle de base reste intact : ce sont deux modèles différents.

Ton token Hugging Face (type **Write**) va t'être demandé.

In [9]:
from getpass import getpass
from huggingface_hub import HfApi, login
from dataclasses import asdict
import shutil

login(token=getpass("Token Hugging Face (Write) : "))

NOM_DEPOT = "Ezeck01/petit-gpt-fr-46m-instruct"
PRIVE = True
SORTIE = f"{TRAVAIL}/instruct"
os.makedirs(SORTIE, exist_ok=True)
save_model(modele.cpu(), f"{SORTIE}/model.safetensors")
modele.to(device)
json.dump(asdict(config), open(f"{SORTIE}/config.json", "w"), indent=2)
tokenizer.save(f"{SORTIE}/tokenizer.json")
shutil.copy(f"{TRAVAIL}/model.py", f"{SORTIE}/model.py")

fiche = f"""---
language: fr
license: apache-2.0
library_name: pytorch
tags: [text-generation, gpt, from-scratch, french, sft]
base_model: {DEPOT}
datasets: [jpacifico/French-Alpaca-dataset-Instruct-110K]
---

# Petit GPT français — version instruct (SFT)

Version ajustée par SFT de [{DEPOT}](https://huggingface.co/{DEPOT}), un GPT de 46 M de paramètres entraîné de zéro sur Wikipédia en français.

## SFT

- Données : French-Alpaca-dataset-Instruct-110K (Jonathan Pacifico, 2024), réponses générées par GPT-3.5-turbo. Exemples contenant du code ou dépassant 512 tokens écartés : {len(train_ex):,} exemples d'entraînement, {len(val_ex):,} de validation.
- Deux tokens spéciaux ajoutés (`<|utilisateur|>`, `<|assistant|>`), initialisés à la moyenne des embeddings existants.
- Perte calculée uniquement sur les réponses (questions masquées).
- {EPOQUES} époques, lots de {TAILLE_LOT} exemples, AdamW, taux d'apprentissage {LR_MAX} avec warmup puis décroissance en cosinus.

Perte sur les réponses de validation : {perte_avant:.3f} avant le SFT, {perte_apres:.3f} après.

## Format

```
<|utilisateur|>
Ta question
<|assistant|>
```
La réponse se termine par `<|endoftext|>`.

## Limites

Le modèle suit le format question → réponse, mais avec 46 M de paramètres ses connaissances sont très limitées : il invente souvent. Projet d'apprentissage, pas un assistant fiable.
"""
open(f"{SORTIE}/README.md", "w", encoding="utf-8").write(fiche)

api = HfApi()
api.create_repo(NOM_DEPOT, private=PRIVE, exist_ok=True)
api.update_repo_settings(NOM_DEPOT, private=PRIVE)
api.upload_folder(folder_path=SORTIE, repo_id=NOM_DEPOT, commit_message="SFT sur French-Alpaca")
print(f"Publié ({'privé' if PRIVE else 'public'}) : https://huggingface.co/{NOM_DEPOT}")

Token Hugging Face (Write) :  ········


Publié (privé) : https://huggingface.co/Ezeck01/petit-gpt-fr-46m-instruct


In [2]:
!pip -q install huggingface_hub
import json
from huggingface_hub import hf_hub_download

chemin = hf_hub_download("jpacifico/French-Alpaca-dataset-Instruct-110K",
                         "Acquiesce_data_110k_instructions.json", repo_type="dataset")
donnees = json.load(open(chemin, encoding="utf-8"))

trouves = [ex for ex in donnees if "Révolution française" in ex["instruction"]]
print(f"{len(trouves)} instructions parlent de la Révolution française\n")
for ex in trouves:
    print("❓", ex["instruction"])
    print("→ ", ex["output"][:200], "\n")

Acquiesce_data_110k_instructions.json: reconstructing file:   0%|          |  0.00B / 52.0MB            

Acquiesce_data_110k_instructions.json: downloading bytes:           |  0.00B            

142 instructions parlent de la Révolution française

❓ Résume en une phrase l'histoire de la Révolution française.
→  "La Révolution française était un événement majeur de la fin du 18e siècle en France, caractérisé par le renversement de la monarchie et l'établissement de la Première République." 

❓ Résume l'événement historique de la Révolution française en 5 points clés.
→  La Révolution française peut être résumée en cinq points clés : 1. L'abolition de la monarchie et l'établissement d'une République 2. L'exécution du roi Louis XVI et de la reine Marie-Antoinette 3. La 

❓ Mets en ordre chronologique les événements suivants : Première Guerre mondiale, Découverte de l'Amérique par Christophe Colomb, Chute de l'Empire romain d'Occident, Révolution française.
→  - Chute de l'Empire romain d'Occident - Découverte de l'Amérique par Christophe Colomb - Révolution française - Première Guerre mondiale 

❓ Complétez la chronologie historique suivante en ajoutant des événements importants 

In [3]:
for ex in donnees:
    if "Révolution française" in ex["instruction"]:
        print(ex["instruction"], "→", ex["output"][:150], "\n")

Résume en une phrase l'histoire de la Révolution française. → "La Révolution française était un événement majeur de la fin du 18e siècle en France, caractérisé par le renversement de la monarchie et l'établisseme 

Résume l'événement historique de la Révolution française en 5 points clés. → La Révolution française peut être résumée en cinq points clés : 1. L'abolition de la monarchie et l'établissement d'une République 2. L'exécution du r 

Mets en ordre chronologique les événements suivants : Première Guerre mondiale, Découverte de l'Amérique par Christophe Colomb, Chute de l'Empire romain d'Occident, Révolution française. → - Chute de l'Empire romain d'Occident - Découverte de l'Amérique par Christophe Colomb - Révolution française - Première Guerre mondiale 

Complétez la chronologie historique suivante en ajoutant des événements importants liés à la Révolution française. → 1789 - Prise de la Bastille 1793 - Exécution de Louis XVI 1799 - Coup d'état de Napoléon Bonaparte 1789 - Décl